# 第 2 周练习 —— 自适应健康伙伴（Adaptive Wellness Companion）

## 练习目标（理念）

把 **对话**、**个性化健康微计划**、**DALL·E 图像** 与 **TTS 语音** 整合进一个 **Gradio** 界面，体验多模态助手怎么串起来。

- **输入**：用户用自然语言描述心情 / 精力 / 压力
- **工具调用（tool calling）**：模型可调用 `get_wellness_plan` 生成结构化计划 JSON
- **输出**：文字回复 + 可选教练语音（TTS）+ 可选灵感海报（图像）

## 和本课第 2 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions + `tools` | `client.chat.completions.create(..., tools=tools)` |
| Function calling 循环 | `finish_reason == "tool_calls"` 时本地执行再回灌 |
| 多模态 | `images.generate`（DALL·E）+ `audio.speech`（TTS） |
| Gradio UI | `gr.Blocks` 把 chatbot / 图 / 音频串成一条事件链 |

## 怎么跑

1. `.env` 里设置 `OPENAI_API_KEY`（缺了会直接 `RuntimeError`）
2. 从上到下运行单元格，最后一格 `launch(inline=True)` 在笔记本内嵌界面
3. 在输入框写近况（例如会议后很累），回车后观察文字 / 音频 / 图片


In [3]:
# ========== 导入：对话、工具、图像、音频、Gradio UI 都要用 ==========

# 标准库 os：读环境变量里的 OPENAI_API_KEY
import os
# 标准库 json：解析 / 序列化 tool 参数与健康计划字典
import json
# 标准库 base64：解码 DALL·E 返回的 b64_json 图片数据
import base64
# BytesIO：把解码后的字节当成「内存文件」给 PIL 打开
from io import BytesIO
# Path：路径工具（本练习主要随其它导入一起保留）
from pathlib import Path
# NamedTemporaryFile：TTS 音频先落到临时 .mp3，再交给 Gradio Audio
from tempfile import NamedTemporaryFile

# OpenAI 官方 SDK：chat / images / audio 都走同一个 client
from openai import OpenAI
# Gradio：快速搭交互界面（Blocks / Chatbot / Image / Audio）
import gradio as gr
# PIL.Image：把字节流转成可展示的图像对象
from PIL import Image
# load_dotenv：从 .env 加载密钥，避免写死在笔记本里
from dotenv import load_dotenv


In [4]:
# ========== 环境检查 + 创建 OpenAI 客户端 ==========

# 加载 .env 到进程环境
load_dotenv()
# 没有 OPENAI_API_KEY 就立刻失败：后面 chat/image/audio 都会依赖它
if not os.getenv("OPENAI_API_KEY"):
    # 错误文案保持英文：这是依赖程序/用户排查的可运行字符串
    raise RuntimeError("Set OPENAI_API_KEY before running the wellness companion.")

# 默认从环境变量读密钥；也可按需传 api_key=
client = OpenAI()


In [5]:
# ========== 模型常量 + system 人设 + tools schema ==========

# 对话主模型：便宜够用的 gpt-4o-mini
MODEL = "gpt-4o-mini"
# 图像模型：DALL·E 3（images.generate 用）
IMAGE_MODEL = "dall-e-3"
# 语音模型：gpt-4o-mini-tts（audio.speech 用）
VOICE_MODEL = "gpt-4o-mini-tts"

# system prompt 与 tools JSON 保持英文原样：改译会改变教练语气 / 工具契约
system_message = (
    "You are an upbeat adaptive wellness coach. "
    "Blend evidence-backed guidance with empathy, tailor plans "
    "to the user's mood, energy, and stress, and explain reasoning concisely."
)

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_wellness_plan",
            "description": "Build a wellness micro-plan keyed to the user's current state.",
            "parameters": {
                "type": "object",
                "properties": {
                    "mood": {"type": "string", "description": "How the user currently feels."},
                    "energy": {"type": "string", "description": "Low, medium, or high energy."},
                    "stress": {"type": "string", "description": "Stress intensity words like calm or overwhelmed."},
                    "focus_goal": {"type": "string", "description": "What the user needs help focusing on right now."}
                },
                "required": ["mood", "energy", "stress", "focus_goal"]
            }
        }
    }
]


In [6]:
# ========== 工具后端：模型决定参数，Python 真正生成计划 JSON ==========

def get_wellness_plan(mood: str, energy: str, stress: str, focus_goal: str) -> str:
    # 统一小写，方便用子串匹配（"High" / "HIGH" 都能命中）
    energy = energy.lower()
    stress = stress.lower()
    # 默认视觉主题 / 动作 / 呼吸 / 反思提示（英文短语会进图像 prompt 与返回 JSON）
    palette = "calming watercolor"
    movement = "gentle mobility flow"
    breathing = "box breathing (4-4-4-4)"
    journaling = "List three small wins and one supportive next step."

    # 精力高：换成更有节奏的步行 + 平衡专注的呼吸
    if "high" in energy:
        movement = "energizing interval walk with posture resets"
        breathing = "alternate nostril breathing to balance focus"
    # 精力低：偏向地面拉伸、减压
    elif "low" in energy:
        movement = "floor-based decompression stretches"

    # 压力大 / 焦虑：更舒缓的配色 + 4-7-8 呼吸降档
    if "over" in stress or "anx" in stress:
        palette = "soothing pastel sanctuary"
        breathing = "4-7-8 breathing to downshift the nervous system"
    # 已经平静：更阳光、积极的视觉主题
    elif "calm" in stress:
        palette = "sunlit studio with optimistic accents"

    # focus_goal 空串时给一个默认焦点词
    focus_goal = focus_goal.strip() or "refocus"

    # 组装结构化计划；visual_theme 后面会喂给 artist() 画图
    plan = {
        "headline": "Adaptive wellness reset",
        "visual_theme": f"{palette} inspired by {mood}",
        "movement": movement,
        "breathing": breathing,
        "reflection": f"Prompt: {journaling}",
        "focus_affirmation": f"Affirmation: You have the capacity to handle {focus_goal} with grace."
    }
    # 返回 JSON 字符串：符合 tool 消息 content 的常见约定
    return json.dumps(plan)

# 注册表：tool 名 → 可调用的 Python 函数（编排层按名字分发）
tool_registry = {"get_wellness_plan": get_wellness_plan}


In [7]:
# ========== 多模态助手：TTS 语音 + DALL·E 图像 ==========

def talker(message: str) -> str | None:
    # 空消息不合成，直接返回 None（Gradio Audio 可接受空）
    if not message:
        return None
    try:
        # with_streaming_response：流式收音频字节，适合稍长的教练回复
        with client.audio.speech.with_streaming_response.create(
            model=VOICE_MODEL,
            # voice 名是 API 枚举值，保持英文
            voice="alloy",
            input=message
        ) as response:
            # delete=False：关闭句柄后文件仍保留，供 Gradio 读取路径
            temp_file = NamedTemporaryFile(suffix=".mp3", delete=False)
            temp_path = temp_file.name
            temp_file.close()
            # 把流式响应写入临时 mp3
            response.stream_to_file(temp_path)
        return temp_path
    except Exception as exc:
        # 音频失败不拖垮主对话：打印警告并返回 None
        print(f"[warn] audio synthesis unavailable: {exc}")
        return None

def artist(theme: str) -> Image.Image | None:
    # 没有视觉主题就不画
    if not theme:
        return None
    try:
        # 图像 prompt 保持英文：直接影响画面风格
        prompt = (
            f"Immersive poster celebrating a wellness ritual, {theme}, "
            "with hopeful lighting and inclusive representation."
        )
        # DALL·E 生成；b64_json 方便在笔记本里直接解码，不走临时 URL
        response = client.images.generate(
            model=IMAGE_MODEL,
            prompt=prompt,
            size="1024x1024",
            response_format="b64_json"
        )
        # 取第一张图的 base64 载荷
        image_base64 = response.data[0].b64_json
        # base64 → bytes → PIL Image
        image_data = base64.b64decode(image_base64)
        return Image.open(BytesIO(image_data))
    except Exception as exc:
        print(f"[warn] image generation unavailable: {exc}")
        return None


In [11]:
# ========== 编排：tool 循环 + 更新历史 + 触发语音/图像 ==========

def handle_tool_calls_and_theme(message) -> tuple[list[dict], str | None]:
    # responses：要追加进 messages 的 role=tool 消息列表
    responses = []
    # theme：从计划 JSON 里抽出 visual_theme，供 artist 使用
    theme = None
    # message.tool_calls 可能为 None，用 or [] 安全迭代
    for tool_call in message.tool_calls or []:
        # 未注册的函数名直接跳过（防御性）
        if tool_call.function.name not in tool_registry:
            continue
        # 模型传来的是 JSON 字符串参数 → 解成 dict
        arguments = json.loads(tool_call.function.arguments)
        # 按名字找到本地函数并 **kwargs 调用
        result = tool_registry[tool_call.function.name](**arguments)
        # 按 API 约定回灌：role=tool + 对应 tool_call_id + content
        responses.append(
            {"role": "tool", "tool_call_id": tool_call.id, "content": result}
        )
        # 再解析一次结果，抓 visual_theme（第一个非空即可）
        payload = json.loads(result)
        theme = theme or payload.get("visual_theme")
    return responses, theme

def chat(history: list[dict]) -> tuple[list[dict], str | None, Image.Image | None]:
    # Gradio messages 格式 → 只保留 role/content 给 Chat Completions
    conversation = [{"role": item["role"], "content": item["content"]} for item in history]
    # 前面拼上 system 人设
    messages = [{"role": "system", "content": system_message}] + conversation
    # 第一次调用：带上 tools，模型可能直接答，也可能请求 tool_calls
    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    theme = None

    # 只要 finish_reason 仍是 tool_calls，就本地执行工具再问模型（可能多轮）
    while response.choices[0].finish_reason == "tool_calls":
        tool_message = response.choices[0].message
        tool_responses, candidate_theme = handle_tool_calls_and_theme(tool_message)
        if candidate_theme:
            theme = candidate_theme
        # 先追加助手那条「我要调工具」的 message，再追加 tool 结果
        messages.append(tool_message)
        messages.extend(tool_responses)
        # 带着工具结果再请求一次
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 最终文本回复
    reply = response.choices[0].message.content
    # 把 assistant 消息追加进聊天历史，供 UI 显示
    updated_history = history + [{"role": "assistant", "content": reply}]
    # 用回复全文做 TTS
    audio_path = talker(reply)
    # 用 tool 抽出的主题画图（可能为 None）
    image = artist(theme)
    # 调试打印：笔记本输出里能看到 Image 对象摘要
    print(image)
    return updated_history, audio_path, image

def put_message_in_chatbot(message: str, history: list[dict]) -> tuple[str, list[dict]]:
    # 清空输入框（返回 ""），并把 user 消息追加到 history
    return "", history + [{"role": "user", "content": message}]


In [12]:
# ========== Gradio Blocks：把输入 → 聊天 → 音频/图像串成事件链 ==========

# title 是窗口/页签标题字符串，保持英文原样
with gr.Blocks(title="Adaptive Wellness Companion") as wellness_ui:
    # 界面说明文案（Gradio 展示字符串，保持原英文）
    gr.Markdown("### Tell me how you are doing and I'll craft a micro-plan.")
    # 一行两列：左对话、右灵感图
    with gr.Row():
        chatbot = gr.Chatbot(height=420, type="messages", label="Conversation")
        image_output = gr.Image(height=420, label="Visual Inspiration")
    # 教练语音；autoplay=True 生成后自动播放
    audio_output = gr.Audio(label="Coach Audio", autoplay=True)
    # 用户输入框；placeholder 提示示例说法
    mood_input = gr.Textbox(label="Share your update", placeholder="e.g. Feeling drained after meetings")

    # submit：先把用户消息放进 chatbot，再 .then 调用 chat 更新三路输出
    mood_input.submit(
        fn=put_message_in_chatbot,
        inputs=[mood_input, chatbot],
        outputs=[mood_input, chatbot]
    ).then(
        fn=chat,
        inputs=chatbot,
        outputs=[chatbot, audio_output, image_output]
    )

# queue：启用排队，避免并发请求打乱状态
wellness_ui.queue()


Gradio Blocks instance: 2 backend functions
-------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x11b3d2850>
 |-<gradio.components.chatbot.Chatbot object at 0x11b3d20d0>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x11b3d2850>
 |-<gradio.components.chatbot.Chatbot object at 0x11b3d20d0>
fn_index=1
 inputs:
 |-<gradio.components.chatbot.Chatbot object at 0x11b3d20d0>
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x11b3d20d0>
 |-<gradio.components.audio.Audio object at 0x11b3d25d0>
 |-<gradio.components.image.Image object at 0x11b3d2210>

In [10]:
# ========== 启动：笔记本内嵌界面（不阻塞后续单元格） ==========

# inline=True：嵌在笔记本里；share=False：不生成公网链接
# prevent_thread_lock=True：不锁死内核线程，方便继续跑别的格子
wellness_ui.launch(inline=True, share=False, prevent_thread_lock=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
